# 04 · Sentence-BERT (all-MiniLM-L6-v2) dense retrieval
**Owner:** Laiba Shaikh · **Proposal model #2:** *Sentence-BERT / Sentence Transformer embeddings for semantic retrieval.*

Same protocol as notebook 03: **ablations/tuning use train claims; dev is used only to report**. Documents are indexed as `title. abstract`; embeddings are L2-normalised, so cosine similarity = dot product.

> ⚠️ Run this notebook on a machine with `sentence-transformers` installed (`pip install -r requirements-retrieval.txt`). The first run downloads the model (~100–130 MB) and encodes 5,183 abstracts (a few minutes on CPU, under a minute on a GPU such as Colab's free T4). Embeddings are cached in `models/`, so later runs are instant.

In [ ]:
import sys, json, time
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from retrieval.data import load_corpus, load_claims, build_qrels
from retrieval.dense import DENSE_MODELS, DenseRetriever, encode_corpus, load_sentence_transformer
from retrieval.evaluate import evaluate_run, save_run, load_run, save_metrics, per_query_first_hit_rank
FIG = ROOT / "results" / "figures"; FIG.mkdir(parents=True, exist_ok=True)

corpus = load_corpus()
ids, texts = [d["id"] for d in corpus], [d["text"] for d in corpus]
by_id = {d["id"]: d for d in corpus}
def split(name):
    cl = load_claims(name); qr = build_qrels(cl)
    return {str(c["id"]): c["claim"] for c in cl if str(c["id"]) in qr}, qr
q_tr, qrels_tr = split("train"); q_dv, qrels_dv = split("dev")
try:
    import torch; print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("torch not importable")

## 1 · Load the model and encode the corpus (cached)

In [ ]:
KEY = "minilm"
cfg = DENSE_MODELS[KEY]
print(cfg)
REBUILD = False                      # set True to force re-encoding

enc = load_sentence_transformer(KEY)
print("embedding dim:", enc.get_sentence_embedding_dimension(), "| max_seq_length:", enc.max_seq_length)

emb_path, _ = DenseRetriever.paths(KEY)
if REBUILD or not emb_path.exists():
    t0 = time.time()
    emb = encode_corpus(enc, texts, batch_size=32)
    DenseRetriever.save_embeddings(KEY, ids, emb)
    print(f"encoded {emb.shape} in {time.time() - t0:.0f}s -> {emb_path}")
retriever = DenseRetriever.from_cache(KEY, enc)
assert retriever.doc_ids == ids, "cached embeddings do not match the current corpus order - rebuild"
E = retriever.doc_embeddings
print("embeddings:", E.shape, E.dtype, "| unit norm:", bool(np.allclose(np.linalg.norm(E, axis=1), 1, atol=1e-3)))

### Sanity check — does the embedding space make sense?
A claim should be (on average) closer to its gold abstract than to a random abstract. If this fails, something is wrong with encoding or ID alignment — better to catch it here than in the metrics.

In [ ]:
rng = np.random.default_rng(0)
qids = list(q_dv)
Q = retriever._encode_queries([q_dv[q] for q in qids])
idx = {d: i for i, d in enumerate(ids)}
gold = np.array([float(Q[i] @ E[idx[sorted(qrels_dv[q])[0]]]) for i, q in enumerate(qids)])
rand = np.array([float(Q[i] @ E[rng.integers(len(ids))]) for i in range(len(qids))])
print(f"mean cosine(claim, gold abstract)   = {gold.mean():.3f}")
print(f"mean cosine(claim, random abstract) = {rand.mean():.3f}")
print(f"claims closer to gold than to a random doc: {(gold > rand).mean():.1%}")
assert gold.mean() > rand.mean()

## 2 · Ablations (on **train**)

**What is tested:** `max_seq_length` — 26.8% of abstracts exceed 256 *words*, and MiniLM's native limit is 256 *tokens*, so text beyond the limit is silently ignored. Does feeding more text help?
(512 is the position-embedding limit of the underlying BERT; 128 is shown for contrast.) Each setting re-encodes the corpus. Set `RUN_ABLATION = False` to skip (~3 encodes).
> If a non-default length wins clearly on **train**, change `max_seq_length` in `retrieval/dense.py::DENSE_MODELS`, set `REBUILD = True` above, and re-run from the top. Do **not** pick the setting from dev.

In [ ]:
RUN_ABLATION = True
if RUN_ABLATION:
    rows = []
    for L in [128, 256, 512]:
        enc.max_seq_length = L
        e = encode_corpus(enc, texts, show_progress_bar=False)
        r = DenseRetriever(f"minilm_L{L}", ids, e, enc, query_prefix=cfg["query_prefix"])
        mm = evaluate_run(r.run(q_tr, 100), qrels_tr)
        rows.append({"max_seq_length": L, "nDCG@10": mm["nDCG@10"], "Recall@10": mm["Recall@10"], "Recall@100": mm["Recall@100"]})
    enc.max_seq_length = cfg["max_seq_length"]       # restore the configured value
    print(pd.DataFrame(rows).round(4).to_string(index=False))

## 3 · Report on dev

In [ ]:
t0 = time.time(); run = retriever.run(q_dv, 100); batch_s = time.time() - t0
m = evaluate_run(run, qrels_dv)
# end-to-end latency for a single live query (what the API sees): encode query + search
sample = list(q_dv.values())[:50]
t0 = time.time()
for q in sample: retriever.search([q], 10)
m["ms_per_query_single"] = 1000 * (time.time() - t0) / len(sample)
m["ms_per_query_batch"] = 1000 * batch_s / len(q_dv)
save_run(KEY, "dev", run)
save_metrics(KEY, "dev", {**m, "config": {**cfg}})

cols = ["Recall@1", "Recall@5", "Recall@10", "Recall@20", "Recall@100", "MRR@10", "nDCG@10", "MAP"]
rows = {KEY: m}
for name in ["tfidf", "bm25"]:                      # produced by notebook 03
    try: rows[name] = json.loads((ROOT / "results" / "metrics" / f"{name}_dev.json").read_text())
    except FileNotFoundError: print("run notebook 03 first to include", name)
print(pd.DataFrame(rows).T[cols].round(3).to_string())
print(f"\n{KEY}: {m['ms_per_query_single']:.1f} ms / live query (CPU/GPU as available)")

## 4 · Error analysis vs. the keyword baseline

In [ ]:
run_bm25 = load_run("bm25", "dev")
r_d = per_query_first_hit_rank(run, qrels_dv)
r_b = per_query_first_hit_rank(run_bm25, qrels_dv)
big = lambda r: 10**6 if r is None else r
dense_wins  = [q for q in r_d if big(r_d[q]) <= 10 < big(r_b[q])]
bm25_wins   = [q for q in r_d if big(r_b[q]) <= 10 < big(r_d[q])]
both_fail   = [q for q in r_d if big(r_d[q]) > 10 and big(r_b[q]) > 10]
print(f"top-10 hit only for {KEY}: {len(dense_wins)} | only for BM25: {len(bm25_wins)} | neither: {len(both_fail)}  (of {len(r_d)})")
dv_claims = {str(c["id"]): c["claim"] for c in load_claims("dev")}
def show(qs, n=4, header=""):
    print("\n" + header)
    for q in qs[:n]:
        g = sorted(qrels_dv[q])[0]
        print(f"  [{q}] {KEY} rank={r_d[q] or '>100'} | BM25 rank={r_b[q] or '>100'}\n     CLAIM: {dv_claims[q]}\n     GOLD : {by_id[g]['title']}")
show(dense_wins, header="Rescued by the dense model (BM25 missed the top-10):")
show(bm25_wins,  header="Won by BM25 (dense model missed the top-10):")
show(both_fail,  header="Both fail:")

In [ ]:
ks = [1, 3, 5, 10, 20, 50, 100]
fig, ax = plt.subplots(figsize=(6.2, 3.8))
for name, r in [(KEY, run), ("bm25", run_bm25), ("tfidf", load_run("tfidf", "dev"))]:
    cur = evaluate_run(r, qrels_dv, ks=ks)
    ax.plot(ks, [cur[f"Recall@{k}"] for k in ks], marker="o", label=name)
ax.set(xscale="log", xlabel="k", ylabel="Recall@k", title=f"{KEY} vs keyword baselines (dev)", ylim=(0.5, 1.0))
ax.set_xticks(ks); ax.set_xticklabels(ks); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig(FIG / f"{KEY}_recall_curve.png", dpi=130); plt.show()

## 5 · Interpretation (fill in after running)
Answer these in 3–4 sentences each — they feed the project's central question *"does semantic retrieval surface more useful evidence than keyword search?"*:
1. Does the dense model beat **tuned BM25** on nDCG@10 / MRR@10 / Recall@10, and by how much? (Report the gap, not just the winner. With only 188 dev queries, differences of ~0.01–0.02 are within noise; consider a paired bootstrap over queries in the evaluation notebook.)
2. In the error analysis, what kind of claims did the dense model rescue, and what did BM25 still win? (vocabulary mismatch vs. exact gene/drug names, claim specificity, abstract length …)
3. Latency vs. quality: is the per-query cost acceptable for the Streamlit demo?